# Notebook 00: Setup & Baseline Evaluation

**Purpose:** Install dependencies, load the base model, evaluate it on the 10 DAA test prompts.

**Inputs:** `data/test_prompts.json`, `data/gold_answers.json` (you must fill this from Claude.ai first - see Step 2 below)

**Outputs:** `results/baseline_results.json` - the base model's responses and BLEU/BERTScore vs gold answers.

**Runtime:** ~10 minutes on a Kaggle T4.

---

## Step 1: Generate gold answers using Claude.ai

Before running this notebook, do the following manually:

1. Open https://claude.ai in your browser
2. Paste this system instruction at the start of a new chat:

> You are a Design and Analysis of Algorithms (DAA) tutor. For each of the questions I send, write a response that **guides the student through the reasoning step by step** rather than just dumping the final answer. Ask leading questions, explain the intuition, build up the solution gradually. The student should learn HOW to think about the problem, not just WHAT the answer is. Keep each response to roughly 200-400 words.

3. For each of the 10 prompts in `data/test_prompts.json`, paste it as a new message and copy Claude's response.
4. Open `data/gold_answers_template.json`, rename it to `gold_answers.json`, and paste each response into the corresponding `gold_answer` field.

You can also send them in batches of 3-4 at a time to save effort - just make sure each response is clearly mapped to its prompt id.


## Step 2: Install dependencies (Kaggle)

In [ ]:
# On Kaggle, this cell installs everything needed. Skip if running locally with deps installed.
import subprocess, sys

PACKAGES = [
    "transformers>=4.45.0",
    "peft>=0.13.0",
    "trl>=0.11.0",
    "bitsandbytes>=0.44.0",
    "accelerate>=1.0.0",
    "datasets>=3.0.0",
    "sacrebleu",
    "bert-score",
    "huggingface_hub",
    "sentencepiece",
    "protobuf"
]
for pkg in PACKAGES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("Dependencies installed.")


## Step 3: Set up paths and HuggingFace authentication

In [ ]:
import os
import sys
import json
from pathlib import Path

# Detect environment: Kaggle vs local
KAGGLE = Path("/kaggle").exists()

if KAGGLE:
    # On Kaggle, clone or copy project files to /kaggle/working
    PROJECT_ROOT = Path("/kaggle/working/daa-helper")
    if not PROJECT_ROOT.exists():
        # If you uploaded the project as a Kaggle dataset, copy it here
        # OR clone from your GitHub repo
        import subprocess
        GITHUB_REPO = "https://github.com/hashirilyas1803/daa-helper.git"
        subprocess.run(["git", "clone", "-b", "instruct-model", GITHUB_REPO, str(PROJECT_ROOT)], check=False)
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

print(f"Project root: {PROJECT_ROOT}")
print(f"Files: {list(PROJECT_ROOT.iterdir())}")


In [ ]:
# HuggingFace login - needed to push model adapters and pull gated models
from huggingface_hub import login

# On Kaggle: store your token as a Kaggle Secret called HF_TOKEN, then:
if KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN") or input("Paste HF token: ").strip()

login(token=HF_TOKEN)
print("Logged in to HF Hub.")

HF_USERNAME = "hashirilyas18"


## Step 4: Load test prompts and gold answers

In [ ]:
from utils.io_helpers import load_json, save_json, save_trial_result

prompts_data = load_json("test_prompts.json", base_dir="data")
gold_data = load_json("gold_answers.json", base_dir="data")  # must exist!

# Validate gold answers are filled in
prompts = [p["prompt"] for p in prompts_data["prompts"]]
gold_answers = [a["gold_answer"] for a in gold_data["answers"]]

unfilled = [i+1 for i, g in enumerate(gold_answers) if "PASTE_CLAUDE" in g or not g.strip()]
if unfilled:
    raise ValueError(f"Gold answers not filled for prompts: {unfilled}. Edit data/gold_answers.json.")

print(f"Loaded {len(prompts)} prompts and {len(gold_answers)} gold answers.")
print("\nSample gold answer (prompt 1):")
print(gold_answers[0][:300] + "...")


## Step 5: Load the base model

We use **Qwen2.5-1.5B-Instruct** as our base. It's small enough to fit comfortably on a T4 with 4-bit quantization (QLoRA setup), has reasonable code/math priors, and is a recent, well-supported model.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit quantization config (QLoRA-style) for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)

print(f"Model loaded: {BASE_MODEL}")
print(f"Device: {model.device}")
print(f"Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## Step 6: Run base model on test prompts

In [ ]:
from utils.evaluation import run_inference_on_prompts, evaluate_responses, free_memory

print("Running inference on 10 test prompts...")
base_responses = run_inference_on_prompts(
    model, tokenizer, prompts,
    max_new_tokens=512,
    temperature=0.7
)

for i, (p, r) in enumerate(zip(prompts, base_responses)):
    print(f"\n--- Prompt {i+1} ---")
    print(f"Q: {p[:80]}...")
    print(f"A: {r[:200]}...")


## Step 7: Score against gold answers

In [ ]:
eval_results = evaluate_responses(base_responses, gold_answers)

print("=" * 60)
print("BASE MODEL EVALUATION")
print("=" * 60)
print(f"Mean BLEU:         {eval_results['aggregate']['mean_bleu']:.2f}")
print(f"Mean BERTScore F1: {eval_results['aggregate']['mean_bertscore_f1']:.4f}")
print(f"Combined score:    {eval_results['aggregate']['combined_score']:.4f}")
print("\nPer-prompt:")
for i, p in enumerate(eval_results["per_prompt"]):
    print(f"  Prompt {i+1}: BLEU={p['bleu']:.2f}, BERTScore F1={p['bertscore_f1']:.4f}")


## Step 8: Save baseline results

In [ ]:
sample_responses = [
    {"prompt": p, "response": r, "gold": g}
    for p, r, g in zip(prompts, base_responses, gold_answers)
]

save_trial_result(
    trial_name="base_model",
    stage="baseline",
    config={"model": BASE_MODEL, "quantization": "4-bit nf4"},
    eval_results=eval_results,
    train_metrics={},
    sample_responses=sample_responses
)

print("\nBaseline saved. Next: run notebook 01 to generate DPO preference pairs.")


## Step 9: (Optional) Push results to HF Hub for safety

This safeguards your results against Kaggle session loss.

In [ ]:
# Uncomment to push results to a HuggingFace dataset repo
# from utils.io_helpers import push_to_hf_hub
# push_to_hf_hub(
#     local_dir=str(PROJECT_ROOT / "results"),
#     repo_id=f"{HF_USERNAME}/daa-helper-results",
#     repo_type="dataset",
#     commit_message="baseline results"
# )

free_memory()
del model
free_memory()
print("\n✓ Notebook 00 complete. Outputs saved to results/baseline_base_model.json")
